# Adversarial Machine Learning: FGSM and PGD Attacks on CIFAR-10

This notebook demonstrates adversarial attacks on a ResNet18 model trained on CIFAR-10.

**CS685 - Advanced Topics in Machine Learning**

## Overview
- **Model**: ResNet18
- **Dataset**: CIFAR-10
- **Attacks**: FGSM (Fast Gradient Sign Method) and PGD (Projected Gradient Descent)

---

In [ ]:
# Import required libraries
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import config
from utils import get_cifar10_loaders, get_model, load_checkpoint
from utils import visualize_adversarial_examples, plot_adversarial_comparison, CIFAR10_CLASSES
from attacks import FGSM, PGD

# Set style
plt.style.use('seaborn-v0_8-darkgrid')

print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {config.DEVICE}")

## 1. Load Model and Data

In [ ]:
# Load data
print("Loading CIFAR-10 dataset...")
train_loader, test_loader = get_cifar10_loaders(batch_size=100)

# Load model
print("\nLoading ResNet18 model...")
model = get_model(num_classes=config.NUM_CLASSES, device=config.DEVICE)

# Try to load trained weights
model_path = '../models/best_model.pth'
try:
    load_checkpoint(model, model_path, device=config.DEVICE)
    print("\nLoaded pre-trained model successfully!")
except:
    print("\nWarning: No pre-trained model found. Using random initialization.")
    print("Please train the model first using: python train.py")

model.eval()
print("\nModel ready for evaluation!")

## 2. Evaluate Clean Accuracy

In [ ]:
def evaluate_accuracy(model, data_loader, device):
    """Evaluate model accuracy"""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in tqdm(data_loader, desc='Evaluating'):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return 100. * correct / total

clean_acc = evaluate_accuracy(model, test_loader, config.DEVICE)
print(f"\nClean Accuracy: {clean_acc:.2f}%")

## 3. FGSM Attack

The Fast Gradient Sign Method (FGSM) is a simple yet effective adversarial attack.

**Formula**: 
$$x_{adv} = x + \epsilon \cdot \text{sign}(\nabla_x J(\theta, x, y))$$

where:
- $x$ is the original input
- $\epsilon$ is the perturbation magnitude
- $J$ is the loss function
- $y$ is the true label

In [ ]:
# FGSM Attack with different epsilon values
epsilon_values = [0, 2/255, 4/255, 8/255, 16/255, 32/255]
fgsm_accuracies = []

print("Evaluating FGSM attack with different epsilon values...\n")

for epsilon in epsilon_values:
    if epsilon == 0:
        fgsm_accuracies.append(clean_acc)
        print(f"ε = {epsilon:.4f}: {clean_acc:.2f}% (Clean)")
        continue
    
    fgsm = FGSM(model=model, epsilon=epsilon)
    correct = 0
    total = 0
    
    for inputs, targets in tqdm(test_loader, desc=f'FGSM ε={epsilon:.4f}'):
        inputs, targets = inputs.to(config.DEVICE), targets.to(config.DEVICE)
        
        # Generate adversarial examples
        adv_inputs = fgsm.generate(inputs, targets)
        
        # Evaluate
        with torch.no_grad():
            outputs = model(adv_inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    accuracy = 100. * correct / total
    fgsm_accuracies.append(accuracy)
    print(f"ε = {epsilon:.4f}: {accuracy:.2f}%")

print(f"\nAccuracy drop: {clean_acc - fgsm_accuracies[-1]:.2f}%")

In [ ]:
# Plot FGSM results
plot_adversarial_comparison(
    clean_acc, 
    fgsm_accuracies, 
    epsilon_values, 
    attack_name='FGSM'
)

## 4. PGD Attack

Projected Gradient Descent (PGD) is an iterative adversarial attack that's more powerful than FGSM.

**Algorithm**:
1. Start from $x^0$ (with random initialization)
2. For $t = 0$ to $T-1$:
   - $x^{t+1} = \Pi_{x + \mathcal{S}}(x^t + \alpha \cdot \text{sign}(\nabla_x J(\theta, x^t, y)))$
3. Return $x^T$

where $\Pi$ projects back to the allowed perturbation set $\mathcal{S}$

In [ ]:
# PGD Attack
epsilon = 8/255
alpha = 2/255
iterations_list = [1, 7, 20, 40]
pgd_accuracies = []

print("Evaluating PGD attack with different iteration counts...\n")

for iterations in iterations_list:
    pgd = PGD(
        model=model,
        epsilon=epsilon,
        alpha=alpha,
        iterations=iterations,
        random_start=True
    )
    
    correct = 0
    total = 0
    
    for inputs, targets in tqdm(test_loader, desc=f'PGD iter={iterations}'):
        inputs, targets = inputs.to(config.DEVICE), targets.to(config.DEVICE)
        
        # Generate adversarial examples
        adv_inputs = pgd.generate(inputs, targets)
        
        # Evaluate
        with torch.no_grad():
            outputs = model(adv_inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    accuracy = 100. * correct / total
    pgd_accuracies.append(accuracy)
    print(f"Iterations = {iterations}: {accuracy:.2f}%")

print(f"\nAccuracy drop (40 iterations): {clean_acc - pgd_accuracies[-1]:.2f}%")

In [ ]:
# Plot PGD results
plt.figure(figsize=(10, 6))
plt.plot(iterations_list, pgd_accuracies, 'o-', linewidth=2, markersize=8, label='PGD Attack', color='red')
plt.axhline(y=clean_acc, color='green', linestyle='--', linewidth=2, label='Clean Accuracy')
plt.xlabel('Number of Iterations', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title(f'PGD Attack Robustness (ε={epsilon:.4f}, α={alpha:.4f})', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)

for i, (iters, acc) in enumerate(zip(iterations_list, pgd_accuracies)):
    plt.annotate(f'{acc:.1f}%', (iters, acc), textcoords="offset points",
                 xytext=(0, 10), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Visualize Adversarial Examples

In [ ]:
# Get a batch of test images
inputs, targets = next(iter(test_loader))
inputs, targets = inputs.to(config.DEVICE), targets.to(config.DEVICE)

# Select first 5 samples
num_samples = 5
inputs = inputs[:num_samples]
targets = targets[:num_samples]

# FGSM Attack
fgsm = FGSM(model=model, epsilon=8/255)
fgsm_adv = fgsm.generate(inputs, targets)

with torch.no_grad():
    fgsm_outputs = model(fgsm_adv)
    _, fgsm_preds = fgsm_outputs.max(1)

print("FGSM Attack Examples:")
visualize_adversarial_examples(
    inputs, fgsm_adv, targets.cpu(), fgsm_preds.cpu(),
    num_samples=num_samples
)

In [ ]:
# PGD Attack
pgd = PGD(model=model, epsilon=8/255, alpha=2/255, iterations=20, random_start=True)
pgd_adv = pgd.generate(inputs, targets)

with torch.no_grad():
    pgd_outputs = model(pgd_adv)
    _, pgd_preds = pgd_outputs.max(1)

print("PGD Attack Examples:")
visualize_adversarial_examples(
    inputs, pgd_adv, targets.cpu(), pgd_preds.cpu(),
    num_samples=num_samples
)

## 6. Compare FGSM vs PGD

In [ ]:
# Summary comparison
print("="*60)
print("ADVERSARIAL ATTACK COMPARISON")
print("="*60)
print(f"Clean Accuracy: {clean_acc:.2f}%")
print()
print(f"FGSM Attack (ε=8/255):")
print(f"  Accuracy: {fgsm_accuracies[3]:.2f}%")
print(f"  Drop: {clean_acc - fgsm_accuracies[3]:.2f}%")
print()
print(f"PGD Attack (ε=8/255, 20 iterations):")
print(f"  Accuracy: {pgd_accuracies[2]:.2f}%")
print(f"  Drop: {clean_acc - pgd_accuracies[2]:.2f}%")
print()
print(f"PGD is stronger by: {fgsm_accuracies[3] - pgd_accuracies[2]:.2f}%")
print("="*60)

## 7. Key Observations

1. **FGSM vs PGD**: PGD is generally more powerful than FGSM as it performs multiple iterative steps.

2. **Epsilon Impact**: Higher epsilon values result in stronger attacks but more visible perturbations.

3. **Iteration Count**: For PGD, more iterations typically lead to stronger attacks.

4. **Imperceptibility**: Even with ε=8/255, perturbations are often imperceptible to humans.

5. **Model Vulnerability**: Standard training makes models highly vulnerable to adversarial attacks.

## 8. Defense Strategies

Potential defenses against adversarial attacks:
- **Adversarial Training**: Train on adversarial examples
- **Defensive Distillation**: Use temperature scaling
- **Input Transformation**: JPEG compression, bit-depth reduction
- **Certified Defenses**: Randomized smoothing, interval bound propagation

---

**References**:
1. Goodfellow et al., "Explaining and Harnessing Adversarial Examples" (2014)
2. Madry et al., "Towards Deep Learning Models Resistant to Adversarial Attacks" (2017)